# B06-00 — 실제 배양 조건·OD600 데이터 확보 및 구조 감사

**목적:** B01–B05의 CTC 기반 단일세포 연구와 독립적으로, 실제 배지 조성과 집단 성장곡선이 연결된 공개 데이터의 획득 가능성과 스키마를 검증한다. **분석 및 최적화는 자료 검증 후 별도 B06 Notebook에서 진행한다.**

- 원본 논문: Aida & Ying, *Scientific Data* 12, 984 (2025), DOI: 10.1038/s41597-025-05356-3
- 자료: https://doi.org/10.6084/m9.figshare.28342064 (Figshare, CC BY 4.0 표기)
- 논문 기술: 1,029개 배지, 13,608개 OD600 성장곡선, 44개 화합물로 배지 설계. 이 값은 **논문 보고치**이며 현재 로컬 자료를 직접 계산한 값이 아니다.

**주의:** OD600은 개별 세포 길이·면적, CFU 또는 세포 생존률과 동일하지 않다. CTC의 train/69 시퀀스와 이 자료 사이에는 확인된 실험적 연결이 없다.

## 1. 실행 환경과 프로젝트 경로
현재 작업 폴더가 `bioimage-ai` 또는 `bioimage-ai/notebooks`인 경우를 지원한다. 새 공개 데이터는 기존 CTC.zip을 덮어쓰지 않는 전용 디렉터리에 저장한다.

In [ ]:
# 입력/출력 폴더 설정 및 환경 확인
from pathlib import Path
import sys, json, hashlib, time
import pandas as pd
import requests

cwd=Path.cwd().resolve()
if cwd.name == 'notebooks':
    root=cwd.parent
elif cwd.name == 'bioimage-ai':
    root=cwd
else:
    raise RuntimeError(f'Jupyter 작업 폴더를 bioimage-ai 또는 notebooks로 바꿔 주세요: {cwd}')
DATA_DIR=root/'data'/'bioculture'/'media_aida_ying_2025'
RESULTS_DIR=root/'results'/'bioculture'/'B06_data_audit'
DATA_DIR.mkdir(parents=True,exist_ok=True)
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
print('Python:',sys.executable)
print('Data:',DATA_DIR)
print('Results:',RESULTS_DIR)

## 2. Figshare 공개 메타데이터 확인
공식 공개 API를 이용해 원본 파일명·파일 크기·다운로드 링크·제공 해시를 조회한다. 실패 시 임의 URL을 구성하지 않는다.

In [ ]:
# 논문에 기록된 Figshare 자료의 메타데이터 조회
ARTICLE_ID=28342064
API_URL=f'https://api.figshare.com/v2/articles/{ARTICLE_ID}'
try:
    response=requests.get(API_URL,timeout=30)
    response.raise_for_status()
except requests.RequestException as ex:
    raise RuntimeError('Figshare API 조회 실패. 인터넷 연결 또는 Figshare 상태를 확인하세요.') from ex
meta=response.json()
files=meta.get('files',[])
if not files: raise RuntimeError('메타데이터에 파일 항목이 없습니다.')
inv=pd.DataFrame([{
    'id':f.get('id'),'name':f.get('name'),'size_bytes':f.get('size'),
    'download_url':f.get('download_url'), 'computed_md5':f.get('computed_md5') or f.get('supplied_md5'),
    'is_link_only':f.get('is_link_only',False)
} for f in files])
inv['size_mb']=inv['size_bytes']/1024**2
inv.to_csv(RESULTS_DIR/'figshare_file_inventory.csv',index=False)
with open(RESULTS_DIR/'figshare_metadata.json','w',encoding='utf-8') as fh:
    json.dump(meta,fh,ensure_ascii=False,indent=2)
print('Title:',meta.get('title'))
print('License:',(meta.get('license') or {}).get('name','확인 필요') if isinstance(meta.get('license'),dict) else meta.get('license'))
print('Files:',len(inv),'Total MB:',round(inv.size_mb.sum(),2))
display(inv[['id','name','size_mb','is_link_only']])

## 3. 안전한 다운로드 선택
처음에는 **목록만 검사**한다. 파일 구성을 확인한 뒤 `DOWNLOAD=True`와 `SELECT_IDS`를 명시해야 다운로드된다. 파일 크기에 상한을 두며 다운로드 중 서버 오류와 해시 불일치를 검사한다. 전체 자료를 무조건 내려받지 않는다.

In [ ]:
# 사용자 확인 뒤에만 실행하도록 기본적으로 다운로드 비활성화
# DOWNLOAD=False
# SELECT_IDS=[]  # 위 표에서 필요한 파일 ID를 복사해 넣으세요. 예: [12345]
# MAX_EACH_MB=200

DOWNLOAD = False
SELECT_IDS = []

if not DOWNLOAD:
    print('다운로드 미실시: 파일 목록을 검토하고 DOWNLOAD=True, SELECT_IDS를 설정하세요.')
else:
    if not SELECT_IDS:
        raise ValueError('SELECT_IDS에 파일 ID를 명시하세요.')
    chosen=inv[inv['id'].isin(SELECT_IDS)]
    if len(chosen)!=len(set(SELECT_IDS)):
        raise ValueError('목록에 없는 ID가 포함되었습니다.')
    for row in chosen.itertuples():
        if row.is_link_only: raise ValueError(f'외부 링크 파일: {row.name}')
        if row.size_mb>MAX_EACH_MB: raise ValueError(f'다운로드 한도 초과: {row.name} ({row.size_mb:.1f}MB)')
        name=Path(row.name).name
        dest=DATA_DIR/name
        if dest.exists():
            print('Already exists, skip:',dest)
            continue
        temp=DATA_DIR/(name+'.part')
        try:
            with requests.get(row.download_url,stream=True,timeout=(20,120)) as r:
                r.raise_for_status()
                with temp.open('wb') as fh:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk: fh.write(chunk)
            if temp.stat().st_size != row.size_bytes:
                raise ValueError(f'크기 불일치: {name}')
            if isinstance(row.computed_md5,str) and len(row.computed_md5)==32:
                h=hashlib.md5()
                with temp.open('rb') as fh:
                    for chunk in iter(lambda:fh.read(1024*1024),b''):h.update(chunk)
                if h.hexdigest().lower()!=row.computed_md5.lower():
                    raise ValueError(f'MD5 불일치: {name}')
            temp.replace(dest)
            print('Downloaded and verified:',dest)
        except Exception:
            print('다운로드 실패 또는 무결성 검사 실패:', name, '중간 파일:',temp)
            raise

## 4. 저장된 표 파일의 시트·열·크기 감사
Excel·CSV 구조를 **읽기 전용**으로 조사한다. 원시 시계열의 관측 시간, Well 반복, 배지 ID, 조성, 미리 계산된 성장 지표를 찾는다. 표의 구조가 확정되기 전에는 JOIN, 결측치 대체, 성장률 추정 등을 실행하지 않는다.

In [ ]:
# 표 구조 탐색: 대용량 파일은 처음 5행만 미리보기
records=[]
for f in sorted(DATA_DIR.iterdir()):
    if f.suffix.lower() in ('.xlsx','.xls'):
        try:
            book=pd.ExcelFile(f)
            for sheet in book.sheet_names:
                preview=pd.read_excel(f,sheet_name=sheet,nrows=5)
                records.append({'file':f.name,'sheet':sheet,'columns':list(map(str,preview.columns)),
                                'preview_rows':len(preview),'sample':preview.head(3).astype(str).to_dict(orient='records')})
        except Exception as e:
            records.append({'file':f.name,'error':str(e)})
    elif f.suffix.lower()=='.csv':
        try:
            preview=pd.read_csv(f,nrows=5)
            records.append({'file':f.name,'sheet':'(csv)','columns':list(map(str,preview.columns)),
                            'preview_rows':len(preview),'sample':preview.head(3).astype(str).to_dict(orient='records')})
        except Exception as e:
            records.append({'file':f.name,'error':str(e)})
with (RESULTS_DIR/'table_schema_audit.json').open('w',encoding='utf-8') as fh:
    json.dump(records,fh,ensure_ascii=False,indent=2)
print('Scanned tables/sheets:',len(records))
for r in records:
    print('\n',r['file'],'|',r.get('sheet','ERROR'))
    print('Columns:',r.get('columns',r.get('error')))

## 5. 품질 확인과 후속 단계 판정
아래 결과를 기록한다.
1. API 목록 확보 여부와 실제 파일 다운로드 성공 여부
2. OD600 원시 기록·배지 조성·실험 반복·품질 정보 각각의 존재 여부
3. 배지 ID·실험 라운드·Well ID 등 연결 키의 확인 여부
4. 데이터 사용 라이선스와 논문 그림의 재사용 조건을 각각 확인

**다음 단계:** 다운받은 파일의 표준화·JOIN·결측치 검사까지 성공한 경우에만 `B06-01` 실제 배지 조건별 성장곡선 분석 Notebook을 작성한다. 이는 CTC 자료와 **독립된 새로운 실험 사례**이다.

In [ ]:
# 검증 파일 목록 정리: 실제 다운로드된 파일만 체크
summary={
 'source_article_id':28342064,
 'api_inventory_exists':(RESULTS_DIR/'figshare_file_inventory.csv').exists(),
 'local_data_files':[{'name':x.name,'bytes':x.stat().st_size} for x in DATA_DIR.iterdir() if x.is_file() and not x.name.endswith('.part')],
 'schema_scanned_items':len(records),
 'ready_for_scientific_analysis':False,
 'next_step':'실제 원시 곡선/조성/품질표 확인 후 연결 키 감사'
}
with (RESULTS_DIR/'acquisition_status.json').open('w',encoding='utf-8') as fh:
 json.dump(summary,fh,ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

In [ ]:
%pip install openpyxl

In [ ]:
import sys
import openpyxl
import pandas as pd

print("Python:", sys.executable)
print("openpyxl:", openpyxl.__version__)
print("pandas:", pd.__version__)